# Urban Context Analysis — Professional GIS Visualization

**Phase 2b · GIS-grade Urban Intelligence**

OSM fetch → NetworkX street graph → centrality → professional 5-figure visualization.

| Figure | File | Description |
|--------|------|-------------|
| 1 | urban_basemap.png | Roads-as-polygons, buildings, trees, frontage heat |
| 2 | urban_network.png | Betweenness + closeness centrality (dark bg) |
| 3 | urban_typology.png | Crossroads/T/Y/roundabout markers + access rings |
| 4 | urban_dashboard.png | 4-panel metrics (frontage, hierarchy, gauge, bar) |
| 5 | urban_response.png | Architectural intelligence text panels |


In [ ]:
import os, sys
from pathlib import Path
import matplotlib
matplotlib.use('Agg')

_cwd = Path(os.getcwd()).resolve()
_root = next(
    (p for p in [_cwd, _cwd.parent, _cwd.parent.parent] if (p / 'team_04').is_dir()),
    None,
)
if _root is None:
    raise RuntimeError('Cannot find team_04/ from ' + str(_cwd))
for _p in [str(_root), str(_root.parent)]:
    if _p not in sys.path:
        sys.path.insert(0, _p)
print('Root:', _root)


In [ ]:
import math, warnings
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D
from shapely.geometry import LineString, Point, Polygon as SlyPoly
from shapely.ops import unary_union

from team_04.agent.tools.osm_context import (
    fetch_context_or_fallback, build_street_graph, INTERESTING_SITES,
)
from team_04.agent.tools.road_context import analyze_roads
from team_04.agent.tools.site_model import build_site_model
from team_04.agent.tools.urban_analysis import (
    full_urban_analysis, compute_centrality, compute_urban_importance,
    classify_intersection_advanced, detect_intersections_from_roads,
    SITE_TYPE_LABELS,
)

plt.rcParams.update({
    'figure.facecolor': '#F4F0E8', 'axes.facecolor': '#F4F0E8',
    'font.family': 'sans-serif', 'font.size': 10,
    'figure.dpi': 130, 'savefig.dpi': 150, 'savefig.bbox': 'tight',
})

PAL = dict(
    bg='#F4F0E8', road_main='#ABABAB', road_sec='#C2C2C2', road_path='#D9D9D9',
    sidewalk='#E3E0D8', res='#E07050', com='#5A8CC0', mixed='#68AE5E',
    civic='#9B72C4', ind='#8E8E8E', unk='#C0B49A', parking='#CFC8B2',
    green='#A2C870', tree_c='#4D8840', site_ec='#D05010', front_hi='#27AE60',
    ix_cross='#8B1A1A', ix_t='#8B3A1A', ix_y='#8B6A1A',
    ix_round='#1A4A8B', ix_dead='#5A5A5A',
)
BTYPE_COLOR = {
    'residential': PAL['res'], 'commercial': PAL['com'],
    'mixed': PAL['mixed'], 'civic': PAL['civic'],
    'industrial': PAL['ind'], 'unknown': PAL['unk'],
}
HIER_FC = {'main': PAL['road_main'], 'secondary': PAL['road_sec'], 'path': PAL['road_path']}
IX_STYLE = {
    'crossroads':       ('X', PAL['ix_cross'], 12),
    't_junction':       ('T', PAL['ix_t'],     11),
    'y_junction':       ('Y', PAL['ix_y'],     11),
    'roundabout':       ('O', PAL['ix_round'], 13),
    'complex_junction': ('*', '#6A0572',        11),
    'dead_end':         ('.', PAL['ix_dead'],   8),
    'bend':             ('.', PAL['ix_dead'],   7),
}
print('Imports OK')


In [ ]:
def _road_poly(road):
    cl = road.get('centerline', []);  w = road.get('width_m', 6.0)
    if len(cl) < 2: return None
    try: return LineString(cl).buffer(w / 2, cap_style=2, join_style=2)
    except: return None

def _sidewalk_poly(road):
    cl = road.get('centerline', []); w = road.get('width_m', 6.0)
    h  = road.get('hierarchy', 'path')
    if len(cl) < 2 or h == 'path': return None
    sw = 3.5 if h == 'main' else 2.5
    try:
        return (LineString(cl).buffer(w/2+sw, cap_style=2, join_style=2)
                .difference(LineString(cl).buffer(w/2, cap_style=2, join_style=2)))
    except: return None

def _draw_poly(ax, geom, fc='#CCC', ec='none', alpha=1.0, lw=0.5, zorder=1):
    if geom is None or geom.is_empty: return
    polys = [geom] if geom.geom_type == 'Polygon' else list(getattr(geom, 'geoms', []))
    for p in polys:
        if hasattr(p, 'exterior'):
            xs, ys = p.exterior.xy
            ax.fill(xs, ys, fc=fc, ec=ec, alpha=alpha, linewidth=lw, zorder=zorder)

def _trees_along_roads(roads, spacing=9.0, view=None):
    out = []
    for road in roads:
        if road.get('hierarchy', 'path') == 'path': continue
        cl = road.get('centerline', [])
        if len(cl) < 2: continue
        line = LineString(cl)
        w = road.get('width_m', 6.0); h = road.get('hierarchy', 'secondary')
        sw = 3.5 if h == 'main' else 2.5; off = w / 2 + sw * 0.55
        n = max(1, int(line.length / spacing))
        for i in range(n):
            t  = (i + 0.4) / n * line.length
            pt = line.interpolate(t)
            t2 = min(t + 0.5, line.length - 0.01)
            p2 = line.interpolate(t2)
            dx, dy = p2.x - pt.x, p2.y - pt.y
            L  = math.hypot(dx, dy) + 1e-9; nx_, ny_ = -dy/L, dx/L
            for s in (1, -1):
                tx, ty = pt.x + s*nx_*off, pt.y + s*ny_*off
                if view and not (view[0]<=tx<=view[2] and view[1]<=ty<=view[3]): continue
                out.append([tx, ty])
    return out

def _gen_context_buildings(roads, cx=0, cy=0, rng=150, step=18):
    from shapely.geometry import box
    rp = [_road_poly(r) for r in roads]; rp = [p for p in rp if p and not p.is_empty]
    if not rp: return []
    ru = unary_union(rp)
    cycle = ['residential','residential','commercial','mixed',
             'residential','unknown','commercial','residential']
    out = []
    for xi, bx in enumerate(range(int(cx-rng), int(cx+rng), step)):
        for yi, by in enumerate(range(int(cy-rng), int(cy+rng), step)):
            m = 1.5; b = box(bx+m, by+m, bx+step-m, by+step-m)
            if ru.intersects(b): continue
            if not (cx-rng < b.centroid.x < cx+rng and cy-rng < b.centroid.y < cy+rng): continue
            out.append({'building_type': cycle[(xi*7+yi*13) % len(cycle)],
                        'polygon_pts': list(b.exterior.coords), 'name': None})
    return out

def _vis_color(s):
    return '#{:02X}{:02X}{:02X}'.format(
        int(244+(42-244)*s), int(226+(157-226)*s), int(133+(143-133)*s))

def _bc_color(s, mn=0, mx=1):
    t = max(0, min(1, (s-mn)/(mx-mn+1e-9)))
    return '#{:02X}{:02X}{:02X}'.format(
        int(37+(212-37)*t), int(99+(37-99)*t), int(212+(37-212)*t))

print('Helpers ready')


In [ ]:
# Swap INTERESTING_SITES index to explore other presets:
#   0 = Eixample Barcelona (crossroads corner)
#   2 = Flatiron New York (triangular corner)
#   3 = Bloomsbury London (T-junction terminal)
SITE_PRESET = INTERESTING_SITES[0]
print(f"Fetching: {SITE_PRESET['name']}")

with warnings.catch_warnings(record=True) as _w:
    warnings.simplefilter('always')
    ctx = fetch_context_or_fallback(
        SITE_PRESET['lat'], SITE_PRESET['lon'],
        radius_m=SITE_PRESET['radius_m'], fallback_index=0,
    )
if _w: print('Offline fallback active -', _w[-1].message)

SOURCE    = ctx['source']
ROADS     = ctx['roads']
BUILDINGS = ctx.get('buildings', [])
PARKING   = ctx.get('parking_areas', [])
GREENS    = ctx.get('green_areas', [])
TREES_OSM = ctx.get('trees', [])
SITE_BDRY = ctx.get('site_boundary', [])
IX_OSM    = ctx.get('intersections', [])

# Phase 2 road analysis
SITE_MODEL   = build_site_model(SITE_BDRY)
ROADS_RESULT = analyze_roads(SITE_MODEL, ROADS)
SITE_MODEL['roads'] = ROADS_RESULT

# Phase 2b urban analysis
URBAN     = full_urban_analysis(SITE_MODEL, ROADS, IX_OSM or None)
FRONTAGES = URBAN['frontages']
ACCESS    = URBAN['access']
CORNERS   = URBAN['corner_conditions']
NEAR_IX   = URBAN['nearby_intersections']
RESPONSE  = URBAN['urban_response']
SITE_TYPE = URBAN['site_type']

# NetworkX street graph + centrality
try:
    G_STREET   = build_street_graph(ROADS)
    CENTRALITY = compute_centrality(G_STREET)
    IMPORTANCE = compute_urban_importance(SITE_MODEL, G_STREET, CENTRALITY)
    NX_OK      = G_STREET is not None and bool(CENTRALITY)
except Exception as _e:
    G_STREET, CENTRALITY, IMPORTANCE, NX_OK = None, {}, {'score':0.5,'grade':'B','factors':{}}, False
    print(f'NetworkX unavailable: {_e}')

# View bounds
all_pts = [p for r in ROADS for p in r.get('centerline', [])] + list(SITE_BDRY)
if all_pts:
    _xs = [p[0] for p in all_pts]; _ys = [p[1] for p in all_pts]
    VIEW_CX  = (min(_xs)+max(_xs))/2; VIEW_CY  = (min(_ys)+max(_ys))/2
    VIEW_RNG = max(max(_xs)-min(_xs), max(_ys)-min(_ys))/2*1.3+25
else:
    VIEW_CX, VIEW_CY, VIEW_RNG = 0, 0, 160
VIEW = (VIEW_CX-VIEW_RNG, VIEW_CY-VIEW_RNG, VIEW_CX+VIEW_RNG, VIEW_CY+VIEW_RNG)

if SOURCE == 'synthetic' and not BUILDINGS:
    BUILDINGS = _gen_context_buildings(ROADS, VIEW_CX, VIEW_CY, VIEW_RNG)
    print(f'Generated {len(BUILDINGS)} synthetic buildings')

# Shared geometry used by all figure cells
_corners  = SITE_MODEL.get('corners', [])
_sides    = SITE_MODEL.get('sides', [])
_rp       = [_road_poly(r) for r in ROADS]
_rp       = [p for p in _rp if p and not p.is_empty]
_road_u   = unary_union(_rp) if _rp else None
_tree_pts = TREES_OSM if TREES_OSM else _trees_along_roads(ROADS, spacing=9.0, view=VIEW)

print(f'Source:{SOURCE} | roads:{len(ROADS)} | bldgs:{len(BUILDINGS)} | ix:{len(IX_OSM)}')
print(f'Site type: {SITE_TYPE_LABELS.get(SITE_TYPE, SITE_TYPE)}')
print(f'Frontages:{len(FRONTAGES)} | Corners:{len(CORNERS)} | Importance:{IMPORTANCE.get("grade","?")}')
if NX_OK: print(f'Graph: {G_STREET.number_of_nodes()} nodes  {G_STREET.number_of_edges()} edges')


In [ ]:
# ============================================================
# Figure 1 - Professional Base Map
# ============================================================
fig, ax = plt.subplots(figsize=(14, 14))
ax.set_aspect('equal'); ax.axis('off')
ax.set_xlim(VIEW[0], VIEW[2]); ax.set_ylim(VIEW[1], VIEW[3])

for ga in GREENS:  # z=1
    pts = ga.get('polygon_pts', [])
    if len(pts) >= 3:
        _draw_poly(ax, SlyPoly([(p[0],p[1]) for p in pts]), fc=PAL['green'], ec='none', alpha=0.55, zorder=1)

for road in ROADS:  # z=2 sidewalks
    _draw_poly(ax, _sidewalk_poly(road), fc=PAL['sidewalk'], ec='none', zorder=2)

for h in ('path', 'secondary', 'main'):  # z=3 road surfaces
    for road in ROADS:
        if road.get('hierarchy') == h:
            _draw_poly(ax, _road_poly(road), fc=HIER_FC[h], ec='none', zorder=3)

for road in ROADS:  # z=4 centre-line dashes
    if road.get('hierarchy') == 'main':
        cl = road.get('centerline', [])
        if len(cl) >= 2:
            ax.plot([p[0] for p in cl], [p[1] for p in cl],
                    color='white', lw=0.7, dashes=(6,5), alpha=0.55, zorder=4)

for bldg in BUILDINGS:  # z=6-7 buildings
    pts = bldg.get('polygon_pts', [])
    if len(pts) < 3: continue
    bp = SlyPoly([(p[0],p[1]) for p in pts])
    if not bp.is_valid: bp = bp.buffer(0)
    if _road_u: bp = bp.difference(_road_u)
    if bp.is_empty: continue
    fc = BTYPE_COLOR.get(bldg.get('building_type', 'unknown'), PAL['unk'])
    _draw_poly(ax, SlyPoly([(p[0]+2,p[1]-2) for p in pts]), fc='#00000014', ec='none', zorder=6)
    _draw_poly(ax, bp, fc=fc, ec='#FFFFFF70', lw=0.7, zorder=7)

for tp in _tree_pts:  # z=9 trees
    if not (VIEW[0]<tp[0]<VIEW[2] and VIEW[1]<tp[1]<VIEW[3]): continue
    if _road_u and _road_u.contains(Point(tp)): continue
    ax.add_patch(plt.Circle((tp[0],tp[1]), 2.4, color=PAL['tree_c'], alpha=0.88, zorder=9, lw=0))

if len(SITE_BDRY) >= 3:  # z=11-12 site boundary
    sp = SlyPoly([(p[0],p[1]) for p in SITE_BDRY]); xs_s, ys_s = sp.exterior.xy
    ax.fill(xs_s, ys_s, color='#FF8C00', alpha=0.13, zorder=11)
    ax.plot(xs_s, ys_s, color=PAL['site_ec'], lw=2.8, dashes=(9,4), zorder=12)

for f in FRONTAGES:  # z=13 frontage heat
    sd = next((s for s in _sides if s.get('edge_index') == f.get('side_index', -1)), None)
    if not sd: continue
    fi = sd.get('from_node_index'); ti = sd.get('to_node_index')
    if fi is None or ti is None or fi >= len(_corners) or ti >= len(_corners): continue
    p1 = _corners[fi]['point']; p2 = _corners[ti]['point']
    ax.plot([p1[0],p2[0]], [p1[1],p2[1]],
            color=_vis_color(f.get('visibility_score', 0.5)),
            lw=6, solid_capstyle='round', alpha=0.9, zorder=13)

for cat, sym, col in [('vehicle','V','#1A3A50'), ('pedestrian','P','#1A6050')]:  # z=14 access
    for ap in ACCESS.get(cat, []):
        pt = ap.get('point', [])
        if len(pt) >= 2:
            ax.text(pt[0], pt[1], sym, ha='center', va='center', fontsize=11, color=col,
                    fontweight='bold', zorder=14,
                    bbox=dict(fc='white', ec=col, lw=1.2, boxstyle='round,pad=0.25', alpha=0.92))

for cc in CORNERS:  # z=15 gateway halos
    pt = cc['point']; col = '#E63946' if cc.get('is_gateway') else PAL['front_hi']
    ax.add_patch(plt.Circle((pt[0],pt[1]), 5+cc['visibility_score']*6,
                             color=col, alpha=0.22, zorder=15, lw=0))
    ax.plot(pt[0], pt[1], 'o', color=col, ms=4, zorder=15)

_labeled = set()  # z=16 street names
for road in ROADS:
    nm = road.get('name'); h = road.get('hierarchy', 'path')
    if not nm or nm in _labeled or h == 'path': continue
    cl = road.get('centerline', [])
    if len(cl) < 2: continue
    mi = len(cl)//2; mx_ = (cl[mi][0]+cl[mi-1][0])/2; my_ = (cl[mi][1]+cl[mi-1][1])/2
    if not (VIEW[0]<mx_<VIEW[2] and VIEW[1]<my_<VIEW[3]): continue
    ang = math.degrees(math.atan2(cl[-1][1]-cl[0][1], cl[-1][0]-cl[0][0]))
    if ang > 90: ang -= 180
    if ang < -90: ang += 180
    ax.text(mx_, my_, nm, ha='center', va='center', fontsize=8 if h=='main' else 6.5,
            rotation=ang, rotation_mode='anchor', color='#2A2A2A', fontweight='bold', zorder=16,
            bbox=dict(fc='#F4F0E8B8', ec='none', pad=0.4))
    _labeled.add(nm)

sb_v = max(10, int(VIEW_RNG/4/10)*10)
sb_x = VIEW[0]+VIEW_RNG*0.12; sb_y = VIEW[1]+VIEW_RNG*0.10
ax.plot([sb_x, sb_x+sb_v], [sb_y,sb_y], color='#333', lw=3, solid_capstyle='butt', zorder=17)
ax.text(sb_x+sb_v/2, sb_y+3, f'{sb_v} m', ha='center', fontsize=8.5, color='#333', fontweight='bold')
na_x = VIEW[2]-VIEW_RNG*0.12; na_y = VIEW[1]+VIEW_RNG*0.10; al = VIEW_RNG*0.07
ax.annotate('', xy=(na_x, na_y+al), xytext=(na_x, na_y),
            arrowprops=dict(arrowstyle='->', color='#222', lw=2.2), zorder=17)
ax.text(na_x, na_y+al+2, 'N', ha='center', fontsize=10, color='#222', fontweight='bold')

leg = [
    mpatches.Patch(color=PAL['road_main'], label='Main road'),
    mpatches.Patch(color=PAL['road_sec'],  label='Secondary road'),
    mpatches.Patch(color=PAL['res'],       label='Residential'),
    mpatches.Patch(color=PAL['com'],       label='Commercial'),
    mpatches.Patch(color=PAL['green'],     label='Green space'),
    mpatches.Patch(color=PAL['tree_c'],    label='Street trees'),
    Line2D([],[],color=_vis_color(1.0), lw=5, label='High-vis frontage'),
    mpatches.Patch(color='#E63946', label='Gateway corner'),
]
ax.legend(handles=leg, loc='upper right', fontsize=8, framealpha=0.93, edgecolor='#CCC')
ax.set_title(
    f"Urban Base Map: {SITE_PRESET['name']}  |  "
    f"{SITE_TYPE_LABELS.get(SITE_TYPE, SITE_TYPE)}  |  {SOURCE.upper()}",
    fontsize=12, fontweight='bold', color='#1A1A2E', pad=10, loc='left',
)
plt.tight_layout(); plt.savefig('urban_basemap.png'); plt.show()
print('Figure 1 saved: urban_basemap.png')


In [ ]:
# ============================================================
# Figure 2 - Street Network Centrality (dark bg)
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(16, 8))
fig.patch.set_facecolor('#1A1A2E')
for ax in axes:
    ax.set_aspect('equal'); ax.axis('off'); ax.set_facecolor('#1A1A2E')
    ax.set_xlim(VIEW[0], VIEW[2]); ax.set_ylim(VIEW[1], VIEW[3])

if NX_OK:
    bc_vals = CENTRALITY.get('betweenness', {})
    cc_vals = CENTRALITY.get('closeness', {})
    max_bc  = max(bc_vals.values(), default=1) or 1

    ax0 = axes[0]
    for road in ROADS: _draw_poly(ax0, _road_poly(road), fc='#2E2E3E', ec='none', zorder=1)
    for u, v in G_STREET.edges():
        sc = (bc_vals.get(u,0) + bc_vals.get(v,0)) / 2
        ax0.plot([G_STREET.nodes[u]['x'], G_STREET.nodes[v]['x']],
                 [G_STREET.nodes[u]['y'], G_STREET.nodes[v]['y']],
                 color=_bc_color(sc, 0, max_bc), lw=1.5+sc/max_bc*4, alpha=0.85, zorder=2)
    for nid, data in G_STREET.nodes(data=True):
        bc_n = bc_vals.get(nid, 0)
        ax0.plot(data['x'], data['y'], 'o',
                 color=_bc_color(bc_n, 0, max_bc), ms=4+bc_n/max_bc*10, alpha=0.9, zorder=3)
    ax0.set_title('Betweenness Centrality', fontsize=11, color='#DDDDEE', fontweight='bold')

    ax1 = axes[1]
    for road in ROADS: _draw_poly(ax1, _road_poly(road), fc='#252535', ec='none', zorder=1)
    for u, v in G_STREET.edges():
        ax1.plot([G_STREET.nodes[u]['x'], G_STREET.nodes[v]['x']],
                 [G_STREET.nodes[u]['y'], G_STREET.nodes[v]['y']],
                 color='#404050', lw=1.0, alpha=0.6, zorder=2)
    for nid, data in G_STREET.nodes(data=True):
        cc_n = cc_vals.get(nid, 0)
        ax1.plot(data['x'], data['y'], 'o',
                 color=_bc_color(cc_n, 0, 1), ms=6+cc_n*14, alpha=0.88, zorder=3)
    ax1.set_title('Closeness Centrality', fontsize=11, color='#DDDDEE', fontweight='bold')
    sc_v = IMPORTANCE.get('score', 0)
    ax1.text(VIEW[2]-5, VIEW[1]+8,
             f"Urban Importance\n{IMPORTANCE.get('grade','?')} ({sc_v:.2f})",
             ha='right', va='bottom', fontsize=12, color='#FFD700', fontweight='bold',
             bbox=dict(fc='#1A1A2E', ec='#FFD700', lw=1.5, boxstyle='round,pad=0.5', alpha=0.9))
else:
    for ax in axes:
        ax.text(0.5, 0.5, 'NetworkX not installed.\npip install networkx',
                ha='center', va='center', fontsize=14, color='#AAAACC', transform=ax.transAxes)

fig.suptitle(f"Street Network: {SITE_PRESET['name']}",
             fontsize=12, fontweight='bold', color='#CCCCEE', y=1.01)
plt.tight_layout(); plt.savefig('urban_network.png'); plt.show()
print('Figure 2 saved: urban_network.png')


In [ ]:
# ============================================================
# Figure 3 - Intersection Typology Map
# ============================================================
ALL_IX = IX_OSM or detect_intersections_from_roads(ROADS)
for ix in ALL_IX:
    if ix.get('degree') == 3 and ix.get('type') in ('t_junction', 'y_junction', None):
        nr = [r for r in ROADS
              if len(r.get('centerline',[])) >= 2
              and LineString(r['centerline']).distance(Point(ix['point'])) < 8]
        is_rb = any(r.get('is_roundabout') for r in nr)
        ix['type'] = classify_intersection_advanced(3, nr, is_roundabout=is_rb)

fig, ax = plt.subplots(figsize=(13, 13))
ax.set_aspect('equal'); ax.axis('off')
ax.set_facecolor('#F0EDE6'); ax.set_xlim(VIEW[0], VIEW[2]); ax.set_ylim(VIEW[1], VIEW[3])

for road in ROADS: _draw_poly(ax, _sidewalk_poly(road), fc=PAL['sidewalk'], ec='none', zorder=1)
for h in ('path','secondary','main'):
    for road in ROADS:
        if road.get('hierarchy') == h:
            _draw_poly(ax, _road_poly(road), fc=HIER_FC[h], ec='none', zorder=2)

if len(SITE_BDRY) >= 3:
    sp = SlyPoly([(p[0],p[1]) for p in SITE_BDRY])
    scx, scy = sp.centroid.x, sp.centroid.y
    for r_d, lbl in [(30,'30 m'), (60,'60 m'), (100,'100 m')]:
        ax.add_patch(plt.Circle((scx,scy), r_d, fill=False, ec='#888', lw=1.2, ls='--', zorder=3))
        ax.text(scx+r_d*0.72, scy+r_d*0.72, lbl, fontsize=7, color='#666', zorder=4)

ix_legend = {}
for ix in ALL_IX:
    pt = ix['point']; deg = ix.get('degree', 3); t = ix.get('type', 't_junction')
    sym, col, fs = IX_STYLE.get(t, ('?','#888',10))
    rm = 5 + min(deg, 8) * 0.9
    ax.add_patch(plt.Circle((pt[0],pt[1]), rm, color=col, alpha=0.18, zorder=5, lw=0))
    ax.plot(pt[0], pt[1], 'o', color=col, ms=rm*0.6, alpha=0.8, zorder=6)
    ax.text(pt[0], pt[1], sym, ha='center', va='center',
            fontsize=fs, color='white', fontweight='bold', zorder=7)
    ax.text(pt[0], pt[1]-rm-1.5, f'd{deg}',
            ha='center', va='top', fontsize=6, color=col, zorder=7)
    ix_legend[t] = col

if len(SITE_BDRY) >= 3:
    sp = SlyPoly([(p[0],p[1]) for p in SITE_BDRY]); xs_s, ys_s = sp.exterior.xy
    ax.fill(xs_s, ys_s, color='#FF8C00', alpha=0.10, zorder=8)
    ax.plot(xs_s, ys_s, color=PAL['site_ec'], lw=2.5, dashes=(8,4), zorder=9)

ix_leg = [Line2D([],[],marker='o',color=c,ls='none',ms=10,
                  label=t.replace('_',' ').title(),markerfacecolor=c,alpha=0.8)
          for t, c in sorted(ix_legend.items())]
if ix_leg: ax.legend(handles=ix_leg, loc='upper right', fontsize=8, title='Intersection Type')

from collections import Counter
_tc = Counter(ix.get('type','?') for ix in ALL_IX)
_summary = 'Total: ' + str(len(ALL_IX))
for t, n in sorted(_tc.items()):
    _summary += '\n' + t.replace('_',' ').title() + ': ' + str(n)
ax.text(VIEW[0]+4, VIEW[3]-4, _summary,
        ha='left', va='top', fontsize=8.5, color='#333',
        bbox=dict(fc='white', ec='#CCC', lw=1, boxstyle='round,pad=0.5', alpha=0.9), zorder=10)

ax.set_title(
    f"Intersection Typology: {SITE_PRESET['name']} | {len(ALL_IX)} junctions",
    fontsize=11, fontweight='bold', color='#1A1A2E', pad=8, loc='left',
)
plt.tight_layout(); plt.savefig('urban_typology.png'); plt.show()
print('Figure 3 saved: urban_typology.png')


In [ ]:
# ============================================================
# Figure 4 - Urban Metrics Dashboard (2x2)
# ============================================================
from collections import Counter
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.patch.set_facecolor(PAL['bg'])

# A: Frontage visibility map
ax_a = axes[0,0]; ax_a.set_aspect('equal'); ax_a.axis('off')
ax_a.set_xlim(VIEW[0], VIEW[2]); ax_a.set_ylim(VIEW[1], VIEW[3])
for road in ROADS: _draw_poly(ax_a, _road_poly(road), fc='#E0DDD8', ec='none', zorder=1)
if len(SITE_BDRY) >= 3:
    sp = SlyPoly([(p[0],p[1]) for p in SITE_BDRY]); xs_s, ys_s = sp.exterior.xy
    ax_a.fill(xs_s, ys_s, color='#FF8C00', alpha=0.10, zorder=2)
    ax_a.plot(xs_s, ys_s, color=PAL['site_ec'], lw=2.2, dashes=(8,4), zorder=6)
for f in FRONTAGES:
    sd = next((s for s in _sides if s.get('edge_index') == f.get('side_index',-1)), None)
    if not sd: continue
    fi = sd.get('from_node_index'); ti = sd.get('to_node_index')
    if fi is None or ti is None or fi >= len(_corners) or ti >= len(_corners): continue
    p1 = _corners[fi]['point']; p2 = _corners[ti]['point']
    vis = f.get('visibility_score', 0.5)
    ax_a.plot([p1[0],p2[0]], [p1[1],p2[1]],
              color=_vis_color(vis), lw=8, solid_capstyle='round', alpha=0.9, zorder=4)
    ax_a.text((p1[0]+p2[0])/2, (p1[1]+p2[1])/2, f'{vis:.0%}',
              ha='center', va='center', fontsize=9, fontweight='bold', color='white', zorder=5,
              bbox=dict(fc='#00000060', ec='none', pad=0.2))
ax_a.set_title('A  Frontage Visibility', fontsize=10, fontweight='bold', color='#1A1A2E', pad=5)

# B: Road hierarchy donut
ax_b = axes[0,1]; ax_b.set_facecolor(PAL['bg'])
hc = Counter(r.get('hierarchy','path') for r in ROADS)
order = ['main','secondary','path']
sizes = [hc.get(k, 0) for k in order]
if any(sizes):
    ax_b.pie(sizes,
             labels=[f"{k.title()} ({hc.get(k,0)})" for k in order],
             autopct='%1.0f%%',
             colors=[HIER_FC[k] for k in order],
             startangle=90, pctdistance=0.7,
             wedgeprops=dict(width=0.5, edgecolor='white', linewidth=1.5),
             textprops=dict(fontsize=9))
ax_b.set_title('B  Road Hierarchy', fontsize=10, fontweight='bold', color='#1A1A2E', pad=5)

# C: Urban importance gauge
ax_c = axes[1,0]; ax_c.set_facecolor(PAL['bg'])
ax_c.set_xlim(-1.3, 1.3); ax_c.set_ylim(-0.3, 1.3)
ax_c.set_aspect('equal'); ax_c.axis('off')
thetas = np.linspace(math.pi, 0, 200)
for i in range(len(thetas)-1):
    t_m = (i+0.5)/(len(thetas)-1)
    ax_c.plot([math.cos(thetas[i]),math.cos(thetas[i+1])],
              [math.sin(thetas[i]),math.sin(thetas[i+1])],
              color=_bc_color(t_m,0,1), lw=25, solid_capstyle='butt', alpha=0.8)
sc_c = IMPORTANCE.get('score', 0.5)
needle = math.pi - sc_c * math.pi
ax_c.annotate('', xy=(math.cos(needle)*0.85, math.sin(needle)*0.85), xytext=(0,0),
              arrowprops=dict(arrowstyle='->', color='#222', lw=3.5, mutation_scale=20))
ax_c.add_patch(plt.Circle((0,0), 0.06, color='#333', zorder=6))
ax_c.text(0, -0.15, f"{IMPORTANCE.get('grade','?')} ({sc_c:.0%})",
          ha='center', fontsize=16, fontweight='bold', color='#1A1A2E')
fac = IMPORTANCE.get('factors', {})
ax_c.text(0, -0.25,
          f"BC={fac.get('betweenness',0):.3f}  CC={fac.get('closeness',0):.3f}  n={fac.get('sample_nodes',0)}",
          ha='center', fontsize=8, color='#555', family='monospace')
ax_c.text(-1.1, -0.02, 'Low', ha='center', fontsize=9, color='#5566AA')
ax_c.text( 1.1, -0.02, 'High', ha='center', fontsize=9, color='#AA3333')
ax_c.set_title('C  Urban Importance', fontsize=10, fontweight='bold', color='#1A1A2E', pad=5)

# D: Frontage profile bar chart
ax_d = axes[1,1]; ax_d.set_facecolor(PAL['bg'])
if FRONTAGES:
    names_d = [f"S{f['side_index']} {(f.get('road_name') or f['road_hierarchy'])[:10]}" for f in FRONTAGES]
    vis_d   = [f['visibility_score'] for f in FRONTAGES]
    ax_d.bar(range(len(FRONTAGES)), vis_d,
             color=[_vis_color(v) for v in vis_d], edgecolor='white', lw=1.2, width=0.6)
    ax_d.set_xticks(list(range(len(FRONTAGES)))); ax_d.set_xticklabels(names_d, fontsize=8.5)
    ax_d.set_ylim(0, 1.15); ax_d.set_ylabel('Visibility Score', fontsize=9)
    ax_d.spines[['top','right']].set_visible(False)
    ax_d.yaxis.grid(True, color='#E0DDD8', zorder=0); ax_d.set_axisbelow(True)
    for i, v in enumerate(vis_d):
        ax_d.text(i, v+0.03, f'{v:.0%}', ha='center', fontsize=8.5, fontweight='bold')
else:
    ax_d.text(0.5,0.5,'No frontages',ha='center',va='center',fontsize=12,transform=ax_d.transAxes)
    ax_d.axis('off')
ax_d.set_title('D  Frontage Profiles', fontsize=10, fontweight='bold', color='#1A1A2E', pad=5)

fig.suptitle(f"Urban Metrics: {SITE_PRESET['name']}",
             fontsize=13, fontweight='bold', color='#1A1A2E', y=1.01)
plt.tight_layout(); plt.savefig('urban_dashboard.png'); plt.show()
print('Figure 4 saved: urban_dashboard.png')


In [ ]:
# ============================================================
# Figure 5 - Architectural Intelligence
# ============================================================
import textwrap
PANEL_DEFS = [
    ('building_response', '#2563D4', '#EFF3FF', 'Urban Response'),
    ('massing_strategy',  '#1A6050', '#EDFAF5', 'Massing Strategy'),
    ('entry_strategy',    '#7E3AA6', '#F5EDFF', 'Entry Strategy'),
    ('facade_strategy',   '#A65C1A', '#FFF5ED', 'Facade Strategy'),
    ('corner_treatment',  '#D42525', '#FFEEED', 'Corner Treatment'),
]

fig = plt.figure(figsize=(16, 9))
fig.patch.set_facecolor(PAL['bg'])

ax_hdr = fig.add_axes([0, 0.88, 1, 0.12])
ax_hdr.set_facecolor('#1A1A2E'); ax_hdr.axis('off')
site_lbl = SITE_TYPE_LABELS.get(SITE_TYPE, SITE_TYPE)
ax_hdr.text(0.02, 0.65, SITE_PRESET['name'], fontsize=14, fontweight='bold',
            color='white', transform=ax_hdr.transAxes, va='center')
ax_hdr.text(0.02, 0.22, f'Site: {site_lbl}', fontsize=11,
            color='#A8D0FF', transform=ax_hdr.transAxes, va='center')
ax_hdr.text(0.98, 0.65,
            f"{len(FRONTAGES)} frontage(s)  {len(CORNERS)} corner(s)  {SOURCE}",
            ha='right', fontsize=10, color='#CCDDFF', transform=ax_hdr.transAxes, va='center')
ax_hdr.text(0.98, 0.22,
            f"Importance: {IMPORTANCE.get('grade','?')} ({IMPORTANCE.get('score',0):.0%})",
            ha='right', fontsize=10, color='#FFD700', fontweight='bold',
            transform=ax_hdr.transAxes, va='center')

ax_site = fig.add_axes([0.01, 0.04, 0.24, 0.82])
ax_site.set_aspect('equal'); ax_site.axis('off'); ax_site.set_facecolor('#ECEADE')
ax_site.set_xlim(VIEW_CX-VIEW_RNG*0.75, VIEW_CX+VIEW_RNG*0.75)
ax_site.set_ylim(VIEW_CY-VIEW_RNG*0.75, VIEW_CY+VIEW_RNG*0.75)
for road in ROADS: _draw_poly(ax_site, _sidewalk_poly(road), fc='#DDDAD2', ec='none', zorder=1)
for h in ('path','secondary','main'):
    for road in ROADS:
        if road.get('hierarchy') == h:
            _draw_poly(ax_site, _road_poly(road), fc=HIER_FC[h], ec='none', zorder=2)
if len(SITE_BDRY) >= 3:
    sp = SlyPoly([(p[0],p[1]) for p in SITE_BDRY]); xs_s, ys_s = sp.exterior.xy
    ax_site.fill(xs_s, ys_s, color='#FF8C00', alpha=0.20, zorder=4)
    ax_site.plot(xs_s, ys_s, color=PAL['site_ec'], lw=2.8, dashes=(7,3), zorder=5)
for f in FRONTAGES:
    sd = next((s for s in _sides if s.get('edge_index') == f.get('side_index',-1)), None)
    if not sd: continue
    fi = sd.get('from_node_index'); ti = sd.get('to_node_index')
    if fi is None or ti is None or fi >= len(_corners) or ti >= len(_corners): continue
    ax_site.plot([_corners[fi]['point'][0], _corners[ti]['point'][0]],
                 [_corners[fi]['point'][1], _corners[ti]['point'][1]],
                 color=_vis_color(f.get('visibility_score',0.5)),
                 lw=5.5, solid_capstyle='round', alpha=0.88, zorder=6)
for cc in CORNERS:
    pt = cc['point']; col = '#E63946' if cc.get('is_gateway') else PAL['front_hi']
    ax_site.plot(pt[0], pt[1], 'o', color=col, ms=7, zorder=7)

active = [(k,ec,fc,t) for k,ec,fc,t in PANEL_DEFS if RESPONSE.get(k)]
pw = 0.70 / max(len(active), 1)
for pi, (key, ec_c, fc_c, title) in enumerate(active):
    text = RESPONSE.get(key, ''); px = 0.27 + pi * pw
    ax_p = fig.add_axes([px, 0.04, pw-0.01, 0.82])
    ax_p.set_facecolor(fc_c); ax_p.axis('off')
    ax_p.add_patch(mpatches.FancyBboxPatch(
        (0, 0.88), 1, 0.12, transform=ax_p.transAxes,
        boxstyle='square,pad=0', fc=ec_c, ec='none', zorder=1))
    ax_p.text(0.5, 0.94, title, ha='center', va='center', fontsize=9.5,
              fontweight='bold', color='white', transform=ax_p.transAxes, zorder=2)
    ax_p.text(0.06, 0.83,
              '\n'.join(textwrap.wrap(text, width=34)),
              ha='left', va='top', fontsize=8.8, color='#1A1A2E', linespacing=1.55,
              transform=ax_p.transAxes, zorder=2)

fig.text(0.5, 0.01, f'Phase 2b  source:{SOURCE}  roads:{len(ROADS)}  buildings:{len(BUILDINGS)}',
         ha='center', fontsize=8, color='#888')
plt.savefig('urban_response.png'); plt.show()
print('Figure 5 saved: urban_response.png')


In [ ]:
from collections import Counter
_all_ix = ALL_IX if 'ALL_IX' in vars() else (IX_OSM or [])
print('='*60)
print(f"  {SITE_PRESET['name']}")
print('='*60)
print(f'  Source     : {SOURCE}')
print(f'  Site type  : {SITE_TYPE_LABELS.get(SITE_TYPE, SITE_TYPE)}')
print(f'  Frontages  : {len(FRONTAGES)}')
print(f'  Gateways   : {sum(1 for c in CORNERS if c.get("is_gateway"))}')
print(f'  Importance : {IMPORTANCE.get("grade","?")} ({IMPORTANCE.get("score",0):.3f})')
print()
for h, n in sorted(Counter(r.get('hierarchy','?') for r in ROADS).items()):
    wm = sum(r.get('width_m',0) for r in ROADS if r.get('hierarchy')==h)/(n or 1)
    print(f'  {h:<12}: {n:3d} roads  avg {wm:.1f} m')
print()
for t, n in sorted(Counter(ix.get('type','?') for ix in _all_ix).items()):
    print(f'  {t:<22}: {n}')
if NX_OK:
    print(f'\n  Graph: {G_STREET.number_of_nodes()} nodes  {G_STREET.number_of_edges()} edges')
print('='*60)
